In [3]:
!pip3 install matplotlib
!pip3 install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import os

In [5]:
!ls ~/projectDir/AbigailData/energy_over_time/energy_decomposition

210421_results	210514_results	210620_results	220518_results
210422_results	210515_results	210622_results	220519_results
210423_results	210606_results	210623_results	220520_results
210425_results	210608_results	220515_results	output_all_reaches
210511_results	210614_results	220516_results	output_all_reaches.zip
210512_results	210619_results	220517_results	summary


In [6]:
folder_of_data = "/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition"

# load files

In [7]:
def find_file_recursive(root_folder, file_name, current_depth=0, max_depth=None, verbose=True):
    """
    Recursively search for a file in all subdirectories
    
    Args:
        root_folder (str): Path to start searching from
        file_name (str): Name of the file to look for
        current_depth (int): Current recursion depth (for display purposes)
        max_depth (int): Maximum depth to search (None for unlimited)
        verbose (bool): Whether to print detailed output
    
    Returns:
        list: List of paths where the file was found
    """
    found_paths = []
    indent = "  " * current_depth
    
    # Check if we've reached max depth
    if max_depth is not None and current_depth > max_depth:
        return found_paths
    
    # Normalize the path to handle different path separators
    root_folder = os.path.normpath(root_folder)
    
    # Check if the folder exists and is accessible
    if not os.path.exists(root_folder):
        if verbose:
            print(f"{indent}❌ Folder does not exist: {root_folder}")
        return found_paths
    
    if not os.path.isdir(root_folder):
        if verbose:
            print(f"{indent}❌ Not a directory: {root_folder}")
        return found_paths
    
    try:
        items = os.listdir(root_folder)
    except PermissionError:
        if verbose:
            print(f"{indent}❌ Permission denied: {root_folder}")
        return found_paths
    except Exception as e:
        if verbose:
            print(f"{indent}❌ Error accessing {root_folder}: {e}")
        return found_paths
    
    if verbose:
        print(f"{indent}📁 Searching in: {root_folder}")
    
    # Check if the file exists in current directory
    file_path = os.path.join(root_folder, file_name)
    if os.path.isfile(file_path):  # Use isfile instead of exists to ensure it's a file
        if verbose:
            print(f"{indent}  ✅ Found '{file_name}'")
        found_paths.append(file_path)
    else:
        if verbose:
            print(f"{indent}  ❌ '{file_name}' not found")
    
    # Recursively search in subdirectories
    subdirs = []
    for item in items:
        item_path = os.path.join(root_folder, item)
        if os.path.isdir(item_path):
            subdirs.append(item_path)
    
    if verbose and subdirs:
        print(f"{indent}  📂 Found {len(subdirs)} subdirectories")
    
    for subdir in subdirs:
        try:
            subfolder_results = find_file_recursive(subdir, file_name, current_depth + 1, max_depth, verbose)
            found_paths.extend(subfolder_results)
        except Exception as e:
            if verbose:
                print(f"{indent}  ❌ Error processing {subdir}: {e}")
    
    return found_paths

In [8]:
def extract_results_folder(file_path):
    """
    Extract the folder name that ends with '_results' from a file path
    
    Args:
        file_path (str): Full path to a file
        
    Returns:
        str: The folder name ending with '_results', or None if not found
    """
    # Split the path into components
    path_parts = file_path.split(os.sep)
    
    # Look for a folder that ends with '_results'
    for part in path_parts:
        if part.endswith('_results'):
            return part
    
    return None

# find all reaches and begin generating images

In [9]:
all_reach_states = find_file_recursive(folder_of_data, "per_reach_state.csv")

📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition
  ❌ 'per_reach_state.csv' not found
  📂 Found 24 subdirectories
  📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results
    ❌ 'per_reach_state.csv' not found
    📂 Found 1 subdirectories
    📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1
      ❌ 'per_reach_state.csv' not found
      📂 Found 4 subdirectories
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1/begin_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results/210421_rep1/mid_reach
        ❌ 'per_reach_state.csv' not found
      📁 Searching in: /data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210421_results

In [10]:
import numpy as np
def plot_single_reach_data(df, output_dir,reach_idx="" , title_prefix="", show_mid_point=True):
    """
    Plot energy and kinematic data (x, y, z coordinates) for individual reaches.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with columns: ['reach_idx', 'stim', 'x', 'y', 'z', 'firing_rate', 'energy', 'j', 'h']
    output_dir : str
        Directory to save plots
    title_prefix : str, optional
        Prefix for plot titles (default="")
    show_mid_point : bool, optional
        Whether to mark the middle point (default=True)
        
    Returns:
    --------
    None
    """
    # Get unique stimuli and reaches
    unique_stims = df['stim'].unique()
    unique_reaches = df['reach_idx'].unique()
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Plot for each stimulus
    for stim in unique_stims:
        stim_data = df[df['stim'] == stim]
        unique_reaches_stim = stim_data['reach_idx'].unique()
        
        # Create figure with 6 subplots: X, Y, Z, Energy, J&H, Firing Rate
        plt.figure(figsize=(14, 18))
        
        # X-coordinate subplot
        plt.subplot(6, 1, 1)
        plt.title(f"{title_prefix} X-Coordinate Over Time, Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['x'], '-', alpha=0.7, label=f'Reach {reach_idx}')
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.ylabel("X Position")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        # Y-coordinate subplot
        plt.subplot(6, 1, 2)
        plt.title(f"{title_prefix} Y-Coordinate Over Time, Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['y'], '-', alpha=0.7, label=f'Reach {reach_idx}')
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.ylabel("Y Position")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        # Z-coordinate subplot
        plt.subplot(6, 1, 3)
        plt.title(f"{title_prefix} Z-Coordinate Over Time, Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['z'], '-', alpha=0.7, label=f'Reach {reach_idx}')
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.ylabel("Z Position")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        # Energy subplot
        plt.subplot(6, 1, 4)
        plt.title(f"{title_prefix} Energy of Neural Activity Over Time, Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['energy'], '-', alpha=0.7, label=f'Reach {reach_idx}')
        
        # Calculate and mark mean energy across all reaches for this stimulus
        mean_energy = stim_data['energy'].mean()
        plt.axhline(y=mean_energy, color='r', linestyle='--',
                   label=f"Mean Energy = {mean_energy:.2f}")
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.ylabel("Energy")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        # J & H values subplot
        plt.subplot(6, 1, 5)
        plt.title(f"{title_prefix} Local Fields (h) and Interactions (j), Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['j'], '-', alpha=0.7, label=f'j - Reach {reach_idx}')
            plt.plot(reach_data['h'], '--', alpha=0.7, label=f'h - Reach {reach_idx}')
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.ylabel("J & H Values")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        # Firing rate subplot
        plt.subplot(6, 1, 6)
        plt.title(f"{title_prefix} Firing Rate Over Time, Stim_{stim}")
        
        for reach_idx in unique_reaches_stim:
            reach_data = stim_data[stim_data['reach_idx'] == reach_idx]
            plt.plot(reach_data['firing_rate'], '-', alpha=0.7, label=f'Reach {reach_idx}')
        
        # Mark midpoint if requested
        if show_mid_point and len(reach_data) > 100:
            mid_point = 400
            plt.axvline(x=mid_point, color='g', linestyle='--', alpha=0.7, label='midpoint')
        
        plt.xlabel("Time")
        plt.ylabel("Firing Rate")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"single_reach_data_{reach_idx}_stim_{stim}.png"), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        # Save individual stimulus data to CSV
        # stim_data.to_csv(os.path.join(output_dir, f"single_reach_data_{reach_idx}_stim_{stim}.csv"), index=False)
    
    print(f"Single reach plots and CSVs saved to {output_dir}")

In [11]:
def ensure_directory_exists(directory_path):
    """
    Check if a directory exists and create it (including all parent directories) if it doesn't.
    
    Args:
        directory_path (str): Path to the directory to check/create
    
    Returns:
        bool: True if directory exists or was created successfully, False if creation failed
    """
    try:
        os.makedirs(directory_path, exist_ok=True)
        return True
    except Exception as e:
        print(f"Error creating directory '{directory_path}': {e}")
        return False

In [12]:

output_dir = "/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/output_all_reaches"

base_reach_stats = pd.read_csv("Energy_over_time_hypothesis.csv")

for reach in all_reach_states[:2]:
    
    if not "full_reach" in reach:
        continue
    
    mouse_id = extract_results_folder(reach).split("_")[0]

    reach_data = pd.read_csv(reach)
    individual_reaches = reach_data.groupby(["reach_idx", "stim"])

    all_reach_output_dir = output_dir + f"/{mouse_id}"
    ensure_directory_exists(all_reach_output_dir)
    
    for name, group_df in individual_reaches:
        print(name)
        plot_single_reach_data(group_df.reset_index(), all_reach_output_dir,reach_idx=name[0] , title_prefix=f"{mouse_id}, index:{str(name[0])}")

print(reach)

/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/mid_reach/per_reach_state.csv


In [66]:
reach

'/data001/projects/enserrog/AbigailData/energy_over_time/energy_decomposition/210423_results/210423_rep1/full_reach/per_reach_state.csv'